In [3]:
import cv2
import numpy as np

In [4]:
def process(path, out_path, size=480):
    img = cv2.imread(path)
    img = cv2.resize(img, (size, size))

    # 1) HSV + threshold
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    lower_blue = np.array([95, 120, 120]) # [179,255,255]
    upper_blue = np.array([130, 255, 255])
    blue = cv2.inRange(hsv, lower_blue, upper_blue)

    # 2) نقفل الفجوات الصغيرة في الخط
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    blue = cv2.morphologyEx(blue, cv2.MORPH_CLOSE, kernel, iterations=2) # closing= dilation + erosion

    # 3) ناخد أكبر contour ونملاه
    contours, _ = cv2.findContours(blue, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mask = np.zeros((size, size), np.uint8)
    if contours:
        biggest = max(contours, key=cv2.contourArea)
        cv2.drawContours(mask, [biggest], -1, 255, thickness=cv2.FILLED)

    # fill holes: لو الخط مقفول، الفلود فيل من الخارج بيسيب الجوه ونضيفه
    ff = mask.copy()
    h, w = ff.shape
    cv2.floodFill(ff, np.zeros((h + 2, w + 2), np.uint8), (0, 0), 255)
    mask = mask | cv2.bitwise_not(ff)

    # 4) نجمع الأصلية + الماسك جنب بعض (480x960)
    mask_bgr = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
    result = np.hstack([img, mask_bgr])  # 3 channel
    cv2.imwrite(out_path, result)
    return result


In [5]:
process(
    r"C:\Users\EL-TAB3EEN\OneDrive\Desktop\4th First term\Computer Vision\lab\Task1 image processing\input1.jpg",
    r"C:\Users\EL-TAB3EEN\OneDrive\Desktop\4th First term\Computer Vision\lab\Task1 image processing\output.jpg"
)


array([[[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        ...,
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        ...,
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        ...,
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]],

       ...,

       [[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        ...,
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        ...,
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]],

       [[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        ...,
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]]], shape=(480, 960, 3), dtype=uint8)